# Deliverable 4 – Data Quality Report

## Objective

This report documents the major data quality issues identified in the
collections dataset.

For each issue, the report covers:

- Detection methodology
- Treatment applied
- Business impact
- Recommended production control

## 4.1 Executive Summary

The dataset contains several data quality issues that can affect recovery
reporting and campaign performance analysis.

The most important issues identified were:

1. Exact duplicate records
2. Duplicate payment records
3. Missing values
4. Timezone inconsistencies
5. Inconsistent agent identifiers
6. Payment attribution sensitivity

These issues were investigated and documented before using the cleaned
Golden Dataset for analysis.

## 4.2 Major Data Quality Issues

| Issue | Detection Method | Treatment | Business Impact |
|---|---|---|---|
| Exact duplicate rows | Compared complete rows using duplicate checks | Removed exact duplicates | Prevents double counting and inflated volumes |
| Duplicate payments | Checked repeated payment IDs and exact duplicate payment records | Kept one copy of each exact duplicate | Duplicate SUCCESS payments can inflate reported recovery |
| Missing values | Calculated null counts by table and column | Retained valid records; missing non-key fields were not automatically rejected | Can reduce completeness of customer and operational analysis |
| Timezone inconsistencies | Compared call timezone with vendor master timezone | Vendor master used as the preferred timezone mapping | Incorrect timezone handling can affect event ordering and attribution |
| Agent identity inconsistency | Compared agent names with agent IDs | Agent ID retained as the identifier; agent name not treated as unique | Can create incorrect agent-level performance attribution |
| Payment attribution sensitivity | Compared payments attributed under 7, 14, 30, and 60-day windows | Attribution window should be explicitly defined | Reported campaign performance can change significantly with the selected window |


## 4.3 Duplicate Records

Exact duplicate rows were found in four tables:

| Table | Raw Rows | Golden Rows | Rows Removed |
|---|---:|---:|---:|
| borrowers | 30,600 | 30,000 | 600 |
| calls | 91,350 | 90,079 | 1,271 |
| payments | 25,500 | 25,014 | 486 |
| whatsapp_events | 60,600 | 60,000 | 600 |

### Business Impact

Duplicate records can inflate transaction volumes and recovery metrics if they
are not removed before analysis.

For payments, 486 extra duplicate rows were removed. These duplicates represented
approximately 1.95% of the raw payment amount.

The Golden Dataset removes exact duplicates while retaining valid records.

## 4.4 Missing Values

Missing values were checked across all analytical tables and columns.

### Major Missing Fields

| Table | Column | Missing Records |
|---|---|---:|
| call_attempts | vendor_id | 2,400 |
| calls | agent_id | 1,827 |
| borrowers | email | 880 |
| borrowers | phone | 604 |
| accounts | borrower_id | 455 |
| payments | payment_reference | 382 |
| field_visits | scheduled_at | 250 |

### Treatment

Missing non-key information was not automatically removed because a missing
value does not necessarily make the complete record invalid.

Payment records with missing payment references were retained because missing
references occurred across different payment statuses.

### Business Impact

Missing values can reduce the completeness of customer, agent, vendor, and
payment-level analysis.

Production pipelines should monitor missing-value rates and alert when they
increase unexpectedly.

## 4.5 Timezone Inconsistency

A major inconsistency was found between the timezone stored in the calls table
and the timezone configured for the corresponding vendor.

| Check | Calls |
|---|---:|
| Total calls | 90,079 |
| Timezone matched vendor master | 30,001 (33.31%) |
| Timezone mismatched vendor master | 60,078 (66.69%) |

### Major Mismatch Patterns

| Stored Timezone | Vendor Timezone | Calls |
|---|---|---:|
| Asia/Kolkata | UTC | 16,068 |
| Asia/Dubai | UTC | 16,004 |
| Asia/Dubai | Asia/Kolkata | 14,015 |
| UTC | Asia/Kolkata | 13,991 |

### Treatment

The vendor master was treated as the preferred source for vendor-level timezone
mapping.

### Business Impact

Incorrect timezone handling can change the chronological ordering of events.
This can affect call-to-payment attribution and therefore campaign performance
analysis.

Production systems should standardize timestamps and store event time with a
clear timezone convention.

## 4.6 Agent Identity Inconsistency

Agent identity was investigated by comparing agent names and agent IDs.

| Metric | Value |
|---|---:|
| Agent records | 30,000 |
| Unique agent IDs | 1,000 |
| Unique agent names | 10 |

Several agent names were associated with a large number of different agent IDs.

Examples:

| Agent Name | Agent IDs |
|---|---:|
| Sneha Das | 958 |
| Priya Mehta | 957 |
| Amit Kumar | 952 |
| Vikram Shah | 936 |

### Treatment

Agent ID should be retained as the operational identifier. Agent name should
not be used as a unique key for performance analysis.

### Business Impact

Using agent name as a unique identifier could combine activity from different
agent IDs and produce incorrect agent-level performance results.

A production system should maintain a stable agent identity mapping table.

## 4.7 Payment Attribution Sensitivity

Payment attribution was investigated by linking each payment to the most recent
call for the same account before the payment.

| Metric | Value |
|---|---:|
| Total payments analysed | 25,014 |
| Payments attributed to a previous call | 17,025 |
| Unattributed payments | 7,989 |
| Average attribution lag | 44 days |
| Median attribution lag | 33 days |

### Attribution Window Sensitivity

| Attribution Window | Payments Within Window |
|---|---:|
| 7 days | 8.94% |
| 14 days | 16.64% |
| 30 days | 31.45% |
| 60 days | 49.67% |

### Treatment

The attribution window should be explicitly defined as part of the business
metric definition. Different windows should not be mixed when comparing
campaign performance.

### Business Impact

Campaign performance can change materially depending on the attribution
window.

A short attribution window may understate campaign impact, while a longer
window may attribute payments that are less directly connected to the
collection activity.

## 4.8 Overall Data Quality Assessment

The main data quality risks are duplicate transactions, missing operational
fields, timezone inconsistencies, unstable agent identifiers, and
attribution-window sensitivity.

These issues can affect recovery reporting, campaign attribution, and
operational performance analysis if they are not controlled.

### Production Recommendations

- Run automated duplicate checks on every ingestion cycle.
- Monitor missing-value rates for important fields.
- Standardize all event timestamps using a consistent timezone convention.
- Maintain stable identity mapping for agents and other operational entities.
- Define and document a single attribution window for campaign reporting.
- Maintain audit logs for corrections and backfills.
- Add automated alerts for unusual changes in row counts, payment amounts,
  recovery rates, and channel mix.

### Final Assessment

The Golden Dataset provides a cleaner and more consistent analytical base,
but production reporting should continue to monitor these data quality risks.

## 4.9 Data Quality Report Conclusion

The data quality assessment identified several issues that could affect
collections analytics and recovery reporting.

The most important risks were duplicate records, missing values, timezone
inconsistencies, agent identity issues, and payment attribution sensitivity.

Exact duplicates were removed while valid records with missing non-key
information were retained. Vendor-level timezone mapping and stable agent
identifiers were recommended for production use.

Overall, the cleaned Golden Dataset provides a stronger foundation for
analysis. However, production reporting should continue to monitor data
quality and apply consistent business rules.